---
## Section 1: Why Sequences? Why Transformer?

### The Key Insight

The strongest bot signal is not WHAT they bid on — it's HOW they bid:

```
Bot pattern:    bid → 0.3s → bid → 0.3s → bid → 0.3s
Human pattern:  bid → 3min → bid → 45min → bid → 2hrs
```

We need to look at bids **in sequence over time**, not just as a summary.

### Why Not Summary Features (Random Forest / MLP)?

We first built a summary approach in Stage 3 (feature_engineering.py) with features like `total_bids`, `mean_time_diff`, `unique_devices`. This works but **loses the temporal pattern**. A bot that bids 500 times with 0.3s gaps looks different from a human who bids 500 times over a week — the sequence tells the story, the summary loses it.

### Why Not LSTM?

We checked actual sequence lengths:
- Median: 18 bids per bidder
- **Max: 515,033 bids per bidder**
- Mean: 1,157 bids per bidder

LSTM reads left-to-right and forgets over very long sequences. With max=515,033, LSTM would forget the earliest bids by the time it reads the latest ones.

### Why Transformer?

- Reads **all bids simultaneously** — no forgetting problem
- Connects bid 1 directly to bid 500 via attention mechanism
- We verified 3,071,224 labelled bid rows exist — enough data for Transformer
- Bot pattern is visible in first 500 bids → we truncate sequences to 500

### Why Also Autoencoder?

After fixing data leakage (Experiment 8), the Transformer's real F1 = 0.25. Root cause: only 103 labelled bot sequences. An autoencoder trains on human sequences only, needing no bot labels at all — it learns 'normal' and flags deviations.


---
## Section 2: Features

### What We Use (8 Features Per Bid)

| Feature | Why It Matters |
|---|---|
| time | Raw timestamp |
| **time_diff** | Gap between consecutive bids — bots bid in 0.3s gaps |
| device | Device fingerprint (5,729 unique values) — bots use same server always |
| country | Bidder country |
| auction | Which auction — bots target same auction repeatedly |
| merchandise | Item category |
| **same_auction_flag** | 1 if same auction as previous bid |
| **same_device_flag** | 1 if same device as previous bid |

### What We Removed and Why

**ip, url** — Too unique per bid. Model would overfit to specific values rather than learning general patterns.

**time_of_day** — Investigated distribution: nearly uniform across all 24 hours. Global platform with different timezones — no day/night pattern. Adding it introduced noise not signal. Removed after Experiment 7.


---
## Section 3: Setup


In [ ]:
!pip install imbalanced-learn -q

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

print('Libraries loaded.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

X = np.load('/content/drive/MyDrive/auction-bot-detector/data/processed/X_sequences.npy')
y = np.load('/content/drive/MyDrive/auction-bot-detector/data/processed/y_labels.npy')

print(f'X shape: {X.shape}  →  (bidders, bids_per_bidder, features)')
print(f'Total bots:   {int(y.sum())} ({int(y.sum())/len(y)*100:.1f}%)')
print(f'Total humans: {int(len(y) - y.sum())} ({(len(y)-int(y.sum()))/len(y)*100:.1f}%)')

### Critical: Split Before Scaling

A common mistake is scaling the entire dataset before splitting. This causes **data leakage** — the scaler learns statistics from the test set, which the model should never see.

We discovered this in Experiment 8. Before fixing it, F1=0.856 (fake). After fixing it, F1=0.25 (real).

Correct order:
1. Split into train/test FIRST
2. Fit scaler on **train only**
3. Transform both sets using the train-fitted scaler


In [ ]:
n_samples, n_steps, n_features = X.shape

# stratify=y ensures same bot ratio in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit on train ONLY — never fit on test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(len(X_train), n_steps, n_features)
X_test  = scaler.transform(X_test.reshape(-1, n_features)).reshape(len(X_test), n_steps, n_features)

print(f'Train: {len(X_train)} bidders ({int(y_train.sum())} bots, {int(len(y_train)-y_train.sum())} humans)')
print(f'Test:  {len(X_test)} bidders ({int(y_test.sum())} bots, {int(len(y_test)-y_test.sum())} humans)')
print(f'X_train mean: {X_train.mean():.4f}  std: {X_train.std():.4f}  (should be ~0 and ~1)')

---
## Section 4: Approach 1 — Transformer Classifier

### Full Experiment Journey

| Exp | Key Change | Bot F1 | Recall | Issue |
|---|---|---|---|---|
| 1 | Baseline | N/A | N/A | nan loss — data not scaled |
| 2 | Fixed scaling + BCE | N/A | 0% | All humans predicted (imbalance) |
| 3 | Weighted loss (18x) | 0.23 | 0.67 | Too many false alarms |
| 4 | Simple oversampling | 0.63 | 0.90 | **Data leakage** |
| 5 | 8 features + Focal Loss | 0.67 | 0.85 | Still leakage |
| 6 | Threshold tuning | 0.675 | 0.85 | Still leakage |
| 7 | time_of_day + scheduler + 50 epochs | 0.856 | 0.890 | Still leakage |
| 8 | Fixed split order + SMOTE | **0.25** | **0.76** | **Honest result** |
| 9 | Overfitting check added | 0.25 | 0.76 | No overfitting confirmed |
| 10 | Removed time_of_day (uniform dist.) | 0.27 | 0.76 | **Final config** |

### Why SMOTE Over Simple Oversampling?

Simple oversampling (resample) **duplicates** existing bot sequences. Model sees same 82 bots 6 times → memorises them → fails on new bots.

SMOTE **generates synthetic sequences** by interpolating between existing bots. More diverse training → model learns general patterns, not specific ones.

**Rule:** SMOTE on train set ONLY. Test set always stays clean.


In [ ]:
# SMOTE on train set ONLY
X_train_2d = X_train.reshape(len(X_train), n_steps * n_features)

print(f'Before SMOTE: Bots={int(y_train.sum())} | Humans={int(len(y_train)-y_train.sum())}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_train_2d, y_train)

X_train_smote = X_resampled.reshape(-1, n_steps, n_features)
y_train_smote = y_resampled

print(f'After SMOTE:  Bots={int(y_train_smote.sum())} | Humans={int(len(y_train_smote)-y_train_smote.sum())}')
print(f'Test set UNCHANGED: {int(y_test.sum())} bots | {int(len(y_test)-y_test.sum())} humans')

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_smote)
X_test_tensor  = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train_smote)
y_test_tensor  = torch.FloatTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=32, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

### Transformer Architecture Decisions

**Step 1 — Embedding (input_dim=8 → d_model=32)**
Without this, `time` (trillions) dominates `device` (0-3). Model ignores device entirely.
Embedding puts all features on equal scale.

**Step 2 — Positional Encoding**
Transformer reads all bids simultaneously — doesn't know which came first.
Adds unique sin/cos signal to each position. LSTM doesn't need this (reads left-to-right).

**Step 3 — Attention (nhead=4)**
4 different attention perspectives look at the sequence simultaneously.
Bot: every bid highly connected to every other (same speed, same device) → high attention.
Human: bids loosely connected → lower attention.

**Step 4 — Classification Head**
Averages across 500 bids → one vector per bidder → Sigmoid → probability.

**Why small model (d_model=32, 1 layer, dropout=0.3)?**
Bigger model (d_model=64, 2 layers) memorised the 82 training bots.
Smaller model + higher dropout forces generalisation.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class BotDetectorTransformer(nn.Module):
    def __init__(self, input_dim=8, d_model=32, nhead=4, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding   = nn.Linear(input_dim, d_model)          # Step 1: equal scale
        self.pos_encoding= PositionalEncoding(d_model)             # Step 2: bid order
        self.transformer = nn.TransformerEncoder(                  # Step 3: attention
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True),
            num_layers=num_layers
        )
        self.classifier  = nn.Sequential(                          # Step 4: Bot/Human
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)        # 8 → 32 dimensions
        x = self.pos_encoding(x)     # add position signals
        x = self.transformer(x)      # attention across 500 bids
        x = x.mean(dim=1)            # average → 1 vector per bidder
        return self.classifier(x).squeeze()  # Bot probability

### Why Focal Loss?

**BCE Loss (Experiment 2):** Treats every example equally. Model gets 94.9% accuracy predicting all humans. Loss decreases even when detecting zero bots.

**Weighted Loss (Experiment 3):** 18x penalty for missing a bot. Helped recall but caused 86% false alarms (precision=0.14).

**Focal Loss (Experiment 5+):** Reduces contribution of easy examples (obvious humans), focuses learning on hard examples (rare bots).
- `alpha=0.75`: upweights bot examples
- `gamma=2`: standard focusing strength

### Why Learning Rate Scheduler?

Fixed lr=0.001 (Experiments 1-4) caused overshooting — model bounced around good weights. Scheduler halves lr every 10 epochs: fast learning early, fine-tuning later.


In [ ]:
class FocalLoss(nn.Module):
    """Focuses learning on hard examples (rare bots).
    Reduces loss for easy examples (obvious humans)."""
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, predictions, targets):
        bce  = nn.BCELoss(reduction='none')(predictions, targets)
        pt   = torch.exp(-bce)               # prob of correct prediction
        return (self.alpha * (1-pt)**self.gamma * bce).mean()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

transformer_model = BotDetectorTransformer().to(device)
criterion = FocalLoss(alpha=0.75, gamma=2)

# weight_decay=0.01: L2 regularisation — penalises large weights
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=0.001, weight_decay=0.01)

# Halves lr every 10 epochs: 0.001 → 0.0005 → 0.00025 → ...
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f'Parameters: {sum(p.numel() for p in transformer_model.parameters()):,}')

In [ ]:
def train_transformer(model, train_loader, test_loader, criterion, optimizer, scheduler, device, epochs=50):
    """Tracks both train AND test loss to detect overfitting.
    Overfitting = train loss keeps falling, test loss rises."""
    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()
        total_train, correct, total = 0, 0, 0

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()            # clear gradients
            preds = model(X_b)               # forward pass
            loss  = criterion(preds, y_b)    # calculate error
            loss.backward()                  # backpropagation
            optimizer.step()                 # update weights
            total_train += loss.item()
            correct += ((preds > 0.5).float() == y_b).sum().item()
            total   += len(y_b)

        model.eval()
        total_test = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                total_test += criterion(model(X_b), y_b).item()

        trl = total_train / len(train_loader)
        tel = total_test  / len(test_loader)
        train_losses.append(trl)
        test_losses.append(tel)
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:2d}/50 | Train: {trl:.4f} | Test: {tel:.4f} | '
                  f'Acc: {correct/total*100:.1f}% | LR: {scheduler.get_last_lr()[0]:.5f}')

    return train_losses, test_losses


t_train_losses, t_test_losses = train_transformer(
    transformer_model, train_loader, test_loader,
    criterion, optimizer, scheduler, device
)

In [ ]:
# Overfitting check
# Small gap = model generalises (not memorising)
# Large gap = overfitting = memorised training, fails on new data
plt.figure(figsize=(10, 5))
plt.plot(range(1, 51), t_train_losses, label='Train Loss', color='blue')
plt.plot(range(1, 51), t_test_losses,  label='Test Loss',  color='red')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Transformer: Train vs Test Loss (Overfitting Check)')
plt.legend(); plt.grid(True); plt.show()

gap = abs(t_train_losses[-1] - t_test_losses[-1])
print(f'Train Loss: {t_train_losses[-1]:.4f} | Test Loss: {t_test_losses[-1]:.4f} | Gap: {gap:.4f}')
print('No overfitting ✅' if gap < 0.05 else ('Slight overfitting ⚠️' if gap < 0.15 else 'Overfitting ❌'))

### Why Threshold Tuning?

Default threshold=0.5 means: predict Bot if probability > 50%. Not always optimal for imbalanced data.

- **Lower threshold (0.3-0.4):** higher recall (catches more bots), more false alarms
- **Higher threshold (0.6-0.7):** fewer false alarms, misses more bots

For auction platforms, **recall > precision** — missing a bot is worse than flagging a real user (who can appeal). We favour threshold=0.4.


In [ ]:
transformer_model.eval()
t_preds_prob, t_labels = [], []

with torch.no_grad():
    for X_b, y_b in test_loader:
        t_preds_prob.extend(transformer_model(X_b.to(device)).cpu().numpy())
        t_labels.extend(y_b.numpy())

t_preds_prob = np.array(t_preds_prob)
t_labels     = np.array(t_labels)

print('Transformer Threshold Analysis:')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 48)

for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (t_preds_prob > thresh).astype(float)
    print(f'{thresh:<12} {precision_score(t_labels, p, zero_division=0):<12.3f} '
          f'{recall_score(t_labels, p, zero_division=0):<12.3f} '
          f'{f1_score(t_labels, p, zero_division=0):<12.3f}')

In [ ]:
# Final result — threshold=0.4 chosen for best recall
print('Transformer Final Results (threshold=0.4):')
print(classification_report(t_labels, (t_preds_prob > 0.4).astype(float),
      target_names=['Human', 'Bot']))

---
## Section 5: Approach 2 — Autoencoder (Anomaly Detection)

### Why We Tried This

After fixing data leakage (Experiment 8), Transformer F1 = 0.25. Root cause: only 103 labelled bot sequences. No amount of tuning fixes fundamental data scarcity.

**Autoencoder approach:** Learn what 'normal human bidding' looks like. Flag deviations as bots. Needs zero bot labels.

**Training:** Human sequences only (1,505 available — 18x more than bot sequences)

**Detection:** High reconstruction error = pattern never seen during training = likely bot

### Why Masked MSE?

Median bidder has only 18 bids. Most of the 500 positions are padding zeros. Standard MSE wastes model capacity reconstructing trivial zeros. Masked MSE ignores padding, focuses on real bid positions.


In [ ]:
# Train on HUMANS ONLY — TensorDataset(X, X) because input = target
X_humans = X_train[y_train == 0]
print(f'Training on {len(X_humans)} human sequences (0 bots — intentional)')

ae_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_humans), torch.FloatTensor(X_humans)),
    batch_size=32, shuffle=True
)
print(f'Train batches: {len(ae_loader)}')

In [ ]:
class BidAutoencoder(nn.Module):
    """Learns to reconstruct normal human bid sequences.
    High reconstruction error = abnormal pattern = bot."""
    def __init__(self, n_features=8, n_steps=500, bottleneck=32):
        super().__init__()
        self.n_steps = n_steps
        self.encoder = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU())
        self.bottleneck_down = nn.Linear(n_steps * 8, bottleneck)  # compress sequence
        self.bottleneck_up   = nn.Linear(bottleneck, n_steps * 8)  # expand back
        self.decoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, n_features))

    def forward(self, x):
        b = x.shape[0]
        enc  = self.encoder(x)                                         # compress each bid
        comp = self.bottleneck_down(enc.reshape(b, -1))                # compress sequence to 32
        exp  = self.bottleneck_up(comp).reshape(b, self.n_steps, 8)   # expand back
        return self.decoder(exp)                                       # reconstruct bids


def masked_mse(recon, orig):
    """Only calculate loss on real bids, not padding zeros."""
    mask = (orig.abs().sum(dim=2) != 0).float().unsqueeze(2).expand_as(orig)
    return ((recon - orig)**2 * mask).sum() / (mask.sum() + 1e-8)


ae_model     = BidAutoencoder().to(device)
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=0.001, weight_decay=0.01)
ae_scheduler = torch.optim.lr_scheduler.StepLR(ae_optimizer, step_size=10, gamma=0.5)
print(f'Autoencoder parameters: {sum(p.numel() for p in ae_model.parameters()):,}')

In [ ]:
ae_losses = []
for epoch in range(50):
    ae_model.train()
    total = sum(
        (lambda loss: (loss.backward(), ae_optimizer.step(), ae_optimizer.zero_grad(), loss.item())[-1])(
            masked_mse(ae_model(X_b.to(device)), X_b.to(device))
        ) for X_b, _ in ae_loader
    )
    ae_losses.append(total / len(ae_loader))
    ae_scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:2d}/50 | Loss: {ae_losses[-1]:.6f} | LR: {ae_scheduler.get_last_lr()[0]:.5f}')

In [ ]:
# Reconstruction errors for all test sequences
ae_model.eval()
ae_errors = []
with torch.no_grad():
    for i in range(len(X_test)):
        x = torch.FloatTensor(X_test[i:i+1]).to(device)
        ae_errors.append(masked_mse(ae_model(x), x).item())

ae_errors = np.array(ae_errors)
bot_err   = ae_errors[y_test == 1]
human_err = ae_errors[y_test == 0]

print(f'Human error: Mean={human_err.mean():.4f} | Std={human_err.std():.4f}')
print(f'Bot error:   Mean={bot_err.mean():.4f} | Std={bot_err.std():.4f}')
print(f'Bot mean is {bot_err.mean()/human_err.mean():.2f}x higher than Human — separation exists')

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(human_err, bins=50, alpha=0.7, label='Human', color='blue')
plt.hist(bot_err,   bins=50, alpha=0.7, label='Bot',   color='red')
plt.axvline(x=0.7, color='green', linestyle='--', label='Threshold=0.7')
plt.xlabel('Reconstruction Error'); plt.ylabel('Count')
plt.title('Autoencoder: Human vs Bot Reconstruction Error\n(Right shift of red = bots reconstruct worse = detected)')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
print('Autoencoder Threshold Analysis:')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 48)
for thresh in [0.5, 0.7, 0.9, 1.1, 1.3, 1.5]:
    p = (ae_errors > thresh).astype(float)
    print(f'{thresh:<12} {precision_score(y_test, p, zero_division=0):<12.3f} '
          f'{recall_score(y_test, p, zero_division=0):<12.3f} '
          f'{f1_score(y_test, p, zero_division=0):<12.3f}')

---
## Section 6: Final Comparison & Conclusions


In [ ]:
print('=' * 60)
print('FINAL RESULTS COMPARISON')
print('=' * 60)

print('\n--- Transformer (threshold=0.4) ---')
print(classification_report(t_labels, (t_preds_prob > 0.4).astype(float), target_names=['Human', 'Bot']))

print('\n--- Autoencoder (threshold=0.7) ---')
print(classification_report(y_test, (ae_errors > 0.7).astype(float), target_names=['Human', 'Bot']))

### What The Results Mean

Both models achieve: **Recall ~76%, F1 ~0.25**.

**This is a data problem, not a model problem.**

Evidence:
- No overfitting (train/test gap < 0.05) ✅
- Model generalises — it just lacks sufficient bot signal
- Root cause: only **21 real bots in test set**
- All 10 experiments hit the same F1 ceiling (~0.25-0.27)

### What Would Actually Improve This

1. **More labelled bot data** — most direct fix. 500+ real bots would significantly help.
2. **Semi-supervised learning** — use all 7.6M bid rows, not just 3M labelled ones
3. **Rule-based hybrid** — hard rules for obvious bots + ML for subtle cases
4. **Better features** — bid velocity per auction, IP reuse patterns

### 5 Key Lessons

1. **Data leakage is subtle** — oversampling before split inflated F1 from 0.25 to 0.856
2. **Accuracy misleads** — 94.9% accuracy means nothing if detecting zero bots
3. **Investigate features empirically** — time_of_day seemed logical but data showed it was noise
4. **Smaller model sometimes wins** — with limited data, complexity = memorisation
5. **Data beats architecture** — no model complexity fixes fundamental data scarcity


In [ ]:
import os
save_dir = '/content/drive/MyDrive/auction-bot-detector/models'
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'model_state_dict': transformer_model.state_dict(),
    'model_type': 'transformer', 'threshold': 0.4, 'input_dim': 8, 'n_steps': 500,
    'features': ['time','time_diff','device','country','auction','merchandise','same_auction_flag','same_device_flag'],
    'performance': {'bot_recall': 0.76, 'bot_precision': 0.15, 'bot_f1': 0.25}
}, f'{save_dir}/transformer_model.pth')

torch.save({
    'model_state_dict': ae_model.state_dict(),
    'model_type': 'autoencoder', 'threshold': 0.7, 'input_dim': 8, 'n_steps': 500,
    'features': ['time','time_diff','device','country','auction','merchandise','same_auction_flag','same_device_flag'],
    'performance': {'bot_recall': 0.76, 'bot_precision': 0.15, 'bot_f1': 0.25}
}, f'{save_dir}/autoencoder_model.pth')

print(f'Both models saved to: {save_dir}')